In [ ]:
#@markdown ### **Imports**
# diffusion policy import
from typing import Tuple, Sequence, Dict, Union, Optional, Callable
import numpy as np
import math
import torch
import torch.nn as nn
import torchvision
import collections

from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler
from tqdm.auto import tqdm
import wandb
import cv2

# env import
import shapely.geometry as sg
import cv2
import skimage.transform as st
from skvideo.io import vwrite
from IPython.display import Video
import gdown
import os

In [ ]:
#@markdown ### **Dataset**
#@markdown
#@markdown Defines `PushTImageDataset` and helper functions
#@markdown
#@markdown The dataset class
#@markdown - Load data ((image, agent_pos), action) from a zarr storage
#@markdown - Normalizes each dimension of agent_pos and action to [-1,1]
#@markdown - Returns
#@markdown  - All possible segments with length `pred_horizon`
#@markdown  - Pads the beginning and the end of each episode with repetition
#@markdown  - key `image`: shape (obs_hoirzon, 3, 96, 96)
#@markdown  - key `agent_pos`: shape (obs_hoirzon, 2)
#@markdown  - key `action`: shape (pred_horizon, 2)



# normalize data
def get_data_stats(data):
    data = data.reshape(-1,data.shape[-1])
    stats = {
        'min': np.min(data, axis=0),
        'max': np.max(data, axis=0)
    }
    return stats

def normalize_data(data, stats):
    # nomalize to [0,1]
    ndata = (data - stats['min']) / (stats['max'] - stats['min'])
    # normalize to [-1, 1]
    ndata = ndata * 2 - 1
    return ndata

def unnormalize_data(ndata, stats):
    ndata = (ndata + 1) / 2
    data = ndata * (stats['max'] - stats['min']) + stats['min']
    return data

# dataset
class Planning2DDataset(torch.utils.data.Dataset):
    def __init__(self,
                 dataset_path: str,
                 action_horizon = 64, 
                 use_padding=True
                ):
        self.dataset_path = dataset_path
        self.use_padding = use_padding

        # hardcoded in the stats
        self.stats = {
            'min': -5.,
            'max': 5.
        }


        self.file_indices = {}
        # index all npy files in datadirectory:
        idx = 0
        for file in os.listdir(self.dataset_path):
            # check the files which are end with specific extension
            if file.endswith(".npy"):
                # print path name of selected files
                self.file_indices[idx] = file
                idx += 1

        self.file_len = idx

        # idx all traj into equal sized chunks for batch training
        self.indices = self.make_indices(action_horizon)
        print(self.indices.shape)



    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # get the start/end indices for this datapoint
        file_idx, start, end = self.indices[idx]

        traj_fname = self.file_indices[file_idx]
        traj_path = os.path.join(self.dataset_path, traj_fname)
        nsample = {}
        

        traj_array = np.load(traj_path)[:, start:end].T
        traj_array = np.float32(traj_array)
        nsample['action'] = normalize_data(traj_array, self.stats)
        nsample['start'] = np.expand_dims(nsample['action'][0,:], axis=0)
        nsample['goal'] = np.expand_dims(nsample['action'][-1,:], axis=0)


        image_num = traj_fname.split("_")[1]
        image_fnmae = "maze_occu_" + image_num + ".png"
        image_path = os.path.join(self.dataset_path, image_fnmae)
        img_array = cv2.imread(image_path)
        img_array = np.array(img_array, dtype=np.float32)
        img_array = np.moveaxis(img_array, -1, 0)
        nsample['image'] = np.expand_dims(img_array, axis=0)

        return nsample

    def make_indices(self, horizon):
        '''
            makes indices for sampling from dataset;
            each index maps to a datapoint
        '''
        indices = []
        for file_idx in range(self.file_len):
            fname = self.file_indices[file_idx]
            traj_path = os.path.join(self.dataset_path, fname)
            path_length = np.load(traj_path).shape[1]

            max_start = path_length - horizon
            if not self.use_padding:
                max_start = min(max_start, path_length - horizon)
            for start in range(max_start):
                end = start + horizon
                indices.append((file_idx, start, end))
        indices = np.array(indices)
        return indices


In [ ]:
#@markdown ### **Dataset Demo**

# download demonstration data from Google Drive
dataset_path = "../data_20230908-113959_10k_maps_100_traj/"
# dataset_path = "../data_20230904-170119/"

# parameters
obs_horizon = 1
action_horizon = 160

# create dataset from file
dataset = Planning2DDataset(
    dataset_path=dataset_path,
    action_horizon = action_horizon
)

# save training data statistics (min, max) for each dim
stats = dataset.stats

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=1000,
    num_workers=24,
    shuffle=True,
    # accelerate cpu-gpu transfer
    pin_memory=True,
    # don't kill worker process afte each epoch
    persistent_workers=True
)

# visualize data in batch
batch = next(iter(dataloader))
print("batch['image'].shape:", batch['image'].shape)
print("batch['action'].shape", batch['action'].shape)
print("batch['start'].shape:", batch['start'].shape)
print("batch['goal'].shape:", batch['goal'].shape)


In [ ]:
#@markdown ### **Network**
#@markdown
#@markdown Defines a 1D UNet architecture `ConditionalUnet1D`
#@markdown as the noies prediction network
#@markdown
#@markdown Components
#@markdown - `SinusoidalPosEmb` Positional encoding for the diffusion iteration k
#@markdown - `Downsample1d` Strided convolution to reduce temporal resolution
#@markdown - `Upsample1d` Transposed convolution to increase temporal resolution
#@markdown - `Conv1dBlock` Conv1d --> GroupNorm --> Mish
#@markdown - `ConditionalResidualBlock1D` Takes two inputs `x` and `cond`. \
#@markdown `x` is passed through 2 `Conv1dBlock` stacked together with residual connection.
#@markdown `cond` is applied to `x` with [FiLM](https://arxiv.org/abs/1709.07871) conditioning.

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb


class Downsample1d(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv1d(dim, dim, 3, 2, 1)

    def forward(self, x):
        return self.conv(x)

class Upsample1d(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.ConvTranspose1d(dim, dim, 4, 2, 1)

    def forward(self, x):
        return self.conv(x)


class Conv1dBlock(nn.Module):
    '''
        Conv1d --> GroupNorm --> Mish
    '''

    def __init__(self, inp_channels, out_channels, kernel_size, n_groups=8):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv1d(inp_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.GroupNorm(n_groups, out_channels),
            nn.Mish(),
        )

    def forward(self, x):
        return self.block(x)


class ConditionalResidualBlock1D(nn.Module):
    def __init__(self,
            in_channels,
            out_channels,
            cond_dim,
            kernel_size=3,
            n_groups=8):
        super().__init__()

        self.blocks = nn.ModuleList([
            Conv1dBlock(in_channels, out_channels, kernel_size, n_groups=n_groups),
            Conv1dBlock(out_channels, out_channels, kernel_size, n_groups=n_groups),
        ])

        # FiLM modulation https://arxiv.org/abs/1709.07871
        # predicts per-channel scale and bias
        cond_channels = out_channels * 2
        self.out_channels = out_channels
        self.cond_encoder = nn.Sequential(
            nn.Mish(),
            nn.Linear(cond_dim, cond_channels),
            nn.Unflatten(-1, (-1, 1))
        )

        # make sure dimensions compatible
        self.residual_conv = nn.Conv1d(in_channels, out_channels, 1) \
            if in_channels != out_channels else nn.Identity()

    def forward(self, x, cond):
        '''
            x : [ batch_size x in_channels x horizon ]
            cond : [ batch_size x cond_dim]

            returns:
            out : [ batch_size x out_channels x horizon ]
        '''
        out = self.blocks[0](x)
        embed = self.cond_encoder(cond)

        embed = embed.reshape(
            embed.shape[0], 2, self.out_channels, 1)
        scale = embed[:,0,...]
        bias = embed[:,1,...]
        out = scale * out + bias

        out = self.blocks[1](out)
        out = out + self.residual_conv(x)
        return out


class ConditionalUnet1D(nn.Module):
    def __init__(self,
        input_dim,
        global_cond_dim,
        diffusion_step_embed_dim=256,
        down_dims=[256,512,1024],
        kernel_size=5,
        n_groups=8
        ):
        """
        input_dim: Dim of actions.
        global_cond_dim: Dim of global conditioning applied with FiLM
          in addition to diffusion step embedding. This is usually obs_horizon * obs_dim
        diffusion_step_embed_dim: Size of positional encoding for diffusion iteration k
        down_dims: Channel size for each UNet level.
          The length of this array determines numebr of levels.
        kernel_size: Conv kernel size
        n_groups: Number of groups for GroupNorm
        """

        super().__init__()
        all_dims = [input_dim] + list(down_dims)
        start_dim = down_dims[0]

        dsed = diffusion_step_embed_dim
        diffusion_step_encoder = nn.Sequential(
            SinusoidalPosEmb(dsed),
            nn.Linear(dsed, dsed * 4),
            nn.Mish(),
            nn.Linear(dsed * 4, dsed),
        )
        cond_dim = dsed + global_cond_dim

        in_out = list(zip(all_dims[:-1], all_dims[1:]))
        mid_dim = all_dims[-1]
        self.mid_modules = nn.ModuleList([
            ConditionalResidualBlock1D(
                mid_dim, mid_dim, cond_dim=cond_dim,
                kernel_size=kernel_size, n_groups=n_groups
            ),
            ConditionalResidualBlock1D(
                mid_dim, mid_dim, cond_dim=cond_dim,
                kernel_size=kernel_size, n_groups=n_groups
            ),
        ])

        down_modules = nn.ModuleList([])
        for ind, (dim_in, dim_out) in enumerate(in_out):
            is_last = ind >= (len(in_out) - 1)
            down_modules.append(nn.ModuleList([
                ConditionalResidualBlock1D(
                    dim_in, dim_out, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                ConditionalResidualBlock1D(
                    dim_out, dim_out, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                Downsample1d(dim_out) if not is_last else nn.Identity()
            ]))

        up_modules = nn.ModuleList([])
        for ind, (dim_in, dim_out) in enumerate(reversed(in_out[1:])):
            is_last = ind >= (len(in_out) - 1)
            up_modules.append(nn.ModuleList([
                ConditionalResidualBlock1D(
                    dim_out*2, dim_in, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                ConditionalResidualBlock1D(
                    dim_in, dim_in, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                Upsample1d(dim_in) if not is_last else nn.Identity()
            ]))

        final_conv = nn.Sequential(
            Conv1dBlock(start_dim, start_dim, kernel_size=kernel_size),
            nn.Conv1d(start_dim, input_dim, 1),
        )

        self.diffusion_step_encoder = diffusion_step_encoder
        self.up_modules = up_modules
        self.down_modules = down_modules
        self.final_conv = final_conv

        print("number of parameters: {:e}".format(
            sum(p.numel() for p in self.parameters()))
        )

    def forward(self,
            sample: torch.Tensor,
            timestep: Union[torch.Tensor, float, int],
            global_cond=None):
        """
        x: (B,T,input_dim)
        timestep: (B,) or int, diffusion step
        global_cond: (B,global_cond_dim)
        output: (B,T,input_dim)
        """
        # (B,T,C)
        sample = sample.moveaxis(-1,-2)
        # (B,C,T)

        # 1. time
        timesteps = timestep
        if not torch.is_tensor(timesteps):
            # TODO: this requires sync between CPU and GPU. So try to pass timesteps as tensors if you can
            timesteps = torch.tensor([timesteps], dtype=torch.long, device=sample.device)
        elif torch.is_tensor(timesteps) and len(timesteps.shape) == 0:
            timesteps = timesteps[None].to(sample.device)
        # broadcast to batch dimension in a way that's compatible with ONNX/Core ML
        timesteps = timesteps.expand(sample.shape[0])

        global_feature = self.diffusion_step_encoder(timesteps)

        if global_cond is not None:
            global_feature = torch.cat([
                global_feature, global_cond
            ], axis=-1)

        x = sample
        h = []
        for idx, (resnet, resnet2, downsample) in enumerate(self.down_modules):
            x = resnet(x, global_feature)
            x = resnet2(x, global_feature)
            h.append(x)
            x = downsample(x)

        for mid_module in self.mid_modules:
            x = mid_module(x, global_feature)

        for idx, (resnet, resnet2, upsample) in enumerate(self.up_modules):
            x = torch.cat((x, h.pop()), dim=1)
            x = resnet(x, global_feature)
            x = resnet2(x, global_feature)
            x = upsample(x)

        x = self.final_conv(x)

        # (B,C,T)
        x = x.moveaxis(-1,-2)
        # (B,T,C)
        return x


In [ ]:
#@markdown ### **Vision Encoder**
#@markdown
#@markdown Defines helper functions:
#@markdown - `get_resnet` to initialize standard ResNet vision encoder
#@markdown - `replace_bn_with_gn` to replace all BatchNorm layers with GroupNorm

def get_resnet(name:str, weights=None, **kwargs) -> nn.Module:
    """
    name: resnet18, resnet34, resnet50
    weights: "IMAGENET1K_V1", None
    """
    # Use standard ResNet implementation from torchvision
    func = getattr(torchvision.models, name)
    resnet = func(weights=weights, **kwargs)

    # remove the final fully connected layer
    # for resnet18, the output dim should be 512
    resnet.fc = torch.nn.Identity()
    return resnet


def replace_submodules(
        root_module: nn.Module,
        predicate: Callable[[nn.Module], bool],
        func: Callable[[nn.Module], nn.Module]) -> nn.Module:
    """
    Replace all submodules selected by the predicate with
    the output of func.

    predicate: Return true if the module is to be replaced.
    func: Return new module to use.
    """
    if predicate(root_module):
        return func(root_module)

    bn_list = [k.split('.') for k, m
        in root_module.named_modules(remove_duplicate=True)
        if predicate(m)]
    for *parent, k in bn_list:
        parent_module = root_module
        if len(parent) > 0:
            parent_module = root_module.get_submodule('.'.join(parent))
        if isinstance(parent_module, nn.Sequential):
            src_module = parent_module[int(k)]
        else:
            src_module = getattr(parent_module, k)
        tgt_module = func(src_module)
        if isinstance(parent_module, nn.Sequential):
            parent_module[int(k)] = tgt_module
        else:
            setattr(parent_module, k, tgt_module)
    # verify that all modules are replaced
    bn_list = [k.split('.') for k, m
        in root_module.named_modules(remove_duplicate=True)
        if predicate(m)]
    assert len(bn_list) == 0
    return root_module

def replace_bn_with_gn(
    root_module: nn.Module,
    features_per_group: int=16) -> nn.Module:
    """
    Relace all BatchNorm layers with GroupNorm.
    """
    replace_submodules(
        root_module=root_module,
        predicate=lambda x: isinstance(x, nn.BatchNorm2d),
        func=lambda x: nn.GroupNorm(
            num_groups=x.num_features//features_per_group,
            num_channels=x.num_features)
    )
    return root_module


In [ ]:
#@markdown ### **Network Demo**

# construct ResNet18 encoder
# if you have multiple camera views, use seperate encoder weights for each view.
vision_encoder = get_resnet('resnet18')

# IMPORTANT!
# replace all BatchNorm with GroupNorm to work with EMA
# performance will tank if you forget to do this!
vision_encoder = replace_bn_with_gn(vision_encoder)

# ResNet18 has output dim of 512
vision_feature_dim = 512
start_dim = 2
goal_dim = 2
# observation feature has 514 dims in total per step
obs_dim = vision_feature_dim + start_dim + goal_dim
action_dim = 2

pred_horizon = 160

# create network object
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon
)

# the final arch has 2 parts
nets = nn.ModuleDict({
    'vision_encoder': vision_encoder,
    'noise_pred_net': noise_pred_net
})

# demo
with torch.no_grad():
    # example inputs
    image = torch.zeros((1, obs_horizon,3,100,100))
    start_pos = torch.zeros((1, obs_horizon,2))
    goal_pos = torch.zeros((1, obs_horizon,2))

    # vision encoder
    image_features = nets['vision_encoder'](
        image.flatten(end_dim=1))
    # (2,512)
    image_features = image_features.reshape(*image.shape[:2],-1)
    # (1,2,512)
    obs = torch.cat([image_features, start_pos, goal_pos],  dim=-1)
    # (1,2,516)

    noised_action = torch.randn((1, pred_horizon, action_dim))
    diffusion_iter = torch.zeros((1,))

    # the noise prediction network
    # takes noisy action, diffusion iteration and observation as input
    # predicts the noise added to action
    noise = nets['noise_pred_net'](
        sample=noised_action,
        timestep=diffusion_iter,
        global_cond=obs.flatten(start_dim=1))

    # illustration of removing noise
    # the actual noise removal is performed by NoiseScheduler
    # and is dependent on the diffusion noise schedule
    denoised_action = noised_action - noise

# for this demo, we use DDPMScheduler with 100 diffusion iterations
num_diffusion_iters = 100
noise_scheduler = DDPMScheduler(
    num_train_timesteps=num_diffusion_iters,
    # the choise of beta schedule has big impact on performance
    # we found squared cosine works the best
    beta_schedule='squaredcos_cap_v2',
    # clip output to [-1,1] to improve stability
    clip_sample=True,
    # our network predicts noise (instead of denoised action)
    prediction_type='epsilon'
)

# device transfer
device = torch.device('cuda')
_ = nets.to(device)

In [ ]:
#@markdown ### **Training**
#@markdown
#@markdown Takes about 2.5 hours. If you don't want to wait, skip to the next cell
#@markdown to load pre-trained weights

num_epochs = 100

# Exponential Moving Average
# accelerates training and improves stability
# holds a copy of the model weights
ema = EMAModel(
    model=nets,
    power=0.75)


In [ ]:

# Standard ADAM optimizer
# Note that EMA parametesr are not optimized
optimizer = torch.optim.AdamW(
    params=nets.parameters(),
    lr=1e-4, weight_decay=1e-6)

# Cosine LR schedule with linear warmup
lr_scheduler = get_scheduler(
    name='cosine',
    optimizer=optimizer,
    num_warmup_steps=500,
    num_training_steps=len(dataloader) * num_epochs
)

# configure logging
wandb_run = wandb.init(
    dir= './runs/')
    
wandb.config.update(
    {
        "output_dir": './runs/'
    }
)

with tqdm(range(num_epochs), desc='Epoch') as tglobal:
    # epoch loop
    for epoch_idx in tglobal:
        epoch_loss = list()
        # batch loop
        batch_idx = 0
        with tqdm(dataloader, desc='Batch', leave=False) as tepoch:
            for nbatch in tepoch:
                # data normalized in dataset
                # device transfer
                nimage = nbatch['image'][:,:obs_horizon].to(device)
                naction = nbatch['action'].to(device)
                nstart = nbatch['start'].to(device)
                ngoal = nbatch['goal'].to(device)

                B = nimage.shape[0]

                # encoder vision features
                image_features = nets['vision_encoder'](
                    nimage.flatten(end_dim=1))
                image_features = image_features.reshape(
                    *nimage.shape[:2],-1)
                # (B,obs_horizon,D)

                # concatenate vision feature and low-dim obs
                obs_features = torch.cat([image_features, nstart, ngoal], dim=-1)
                obs_cond = obs_features.flatten(start_dim=1)
                # (B, obs_horizon * obs_dim)

                # sample noise to add to actions
                noise = torch.randn(naction.shape, device=device)

                # sample a diffusion iteration for each data point
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps,
                    (B,), device=device
                ).long()

                # add noise to the clean images according to the noise magnitude at each diffusion iteration
                # (this is the forward diffusion process)
                noisy_actions = noise_scheduler.add_noise(
                    naction, noise, timesteps)

                # predict the noise residual
                noise_pred = noise_pred_net(
                    noisy_actions, timesteps, global_cond=obs_cond)

                # L2 loss
                loss = nn.functional.mse_loss(noise_pred, noise)

                # optimize
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                # step lr scheduler every batch
                # this is different from standard pytorch behavior
                lr_scheduler.step()

                # update Exponential Moving Average of the model weights
                ema.step(nets)

                # logging
                loss_cpu = loss.item()
                epoch_loss.append(loss_cpu)
                tepoch.set_postfix(loss=loss_cpu)

                if (batch_idx + 1) % 1000  == 0:
                    PATH = "./checkpoints/model_logg_epoch_" + str(epoch_idx) + "_batch_" + str(batch_idx) + ".pth"
                    torch.save(nets.state_dict(), PATH)


                if (batch_idx + 1) % 1 == 0:
                    wandb_run.log({'train_loss': loss.item()})
                batch_idx = batch_idx + 1

        if (epoch_idx + 1) % 1  == 0:
            PATH = "./checkpoints/model_logg_epoch_" + str(epoch_idx) + "_batch_" + str(batch_idx) + ".pth"
            torch.save(nets.state_dict(), PATH)
        tglobal.set_postfix(loss=np.mean(epoch_loss))

# Weights of the EMA model
# is used for inference
ema_nets = ema.averaged_model

In [ ]:
#@markdown ### **Loading Pretrained Checkpoint**
#@markdown Set `load_pretrained = True` to load pretrained weights.

load_pretrained = False
if load_pretrained:
  ckpt_path = "./checkpoints/model_logg_epoch_12_batch_258.pth"

  state_dict = torch.load(ckpt_path, map_location='cuda')
  ema_nets = nets
  ema_nets.load_state_dict(state_dict)
  print('Pretrained weights loaded.')
else:
  print("Skipped pretrained weight loading.")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import io
from PIL import Image
# Use Agg backend for canvas
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas

plt.ioff()
def gen_traj_img(map_img, traj_np):
    # invert colour of occupancy map for 
    # better visualisation
    map_img = 1 - map_img

    # convert the trajectory to the 
    # correct corrdinate system and scale
    origin_transform = np.array([-5., -5.]).reshape(1,2)
    traj_scale = 0.1 # 1 pixel = 0.1m
    traj_img = traj_np - np.repeat(origin_transform, len(traj_np[:,0]),axis = 0)
    traj_img = traj_img / traj_scale

    traj_img[traj_img[:] < 0 ] = 0
    traj_img[traj_img[:] >  99 ] = 99

    # plot the trajectory on the map
    fig = plt.figure()
    _ = plt.imshow(map_img, cmap='gray', figure=fig)
    # traj as line
    # plt.plot(traj_img[0,:], traj_img[1,:], 'r', figure=fig)

    # traj as dots 
    c_size = range(0, len(traj_img[:,0]))
    _ = plt.scatter(traj_img[:,0], traj_img[:,1], s=1, c=c_size, cmap='autumn', figure=fig)
    plt.scatter(traj_img[0,0], traj_img[0,1], marker='o', s=20)
    plt.scatter(traj_img[-1,0], traj_img[-1,1], marker='x', s=20)

    plt.axis('off')

    # convert the plot to an image
    # put pixel buffer in numpy array
    canvas = FigureCanvas(fig)
    canvas.draw()
    mat = np.array(canvas.renderer._renderer)
    mat = cv2.cvtColor(mat, cv2.COLOR_RGB2BGR)
    # mat = np.repeat(mat, 3, axis=2)
    return mat

In [ ]:
fpath_map = "./data_20230908-113959_10k_maps_100_traj/maze_occu_0.png"
fpath_traj = "./data_20230908-113959_10k_maps_100_traj/maze_0_traj_1.npy"
# fpath_map = "../data_20230904-170119/maze_occu_0.png"
# fpath_traj = "../data_20230904-170119/maze_0_traj_0.npy"

map_img = np.asarray(plt.imread(fpath_map)).reshape(100,100,1)
map_img = np.repeat(map_img, 3, axis=2)
example_traj = np.load(fpath_traj).transpose(1,0)
traj_img = gen_traj_img(map_img, example_traj)
# plt.show()

In [ ]:
example_traj.shape

In [ ]:
#@markdown ### **Inference**
plt.ioff()
# use same start and end as the example trajectory
# start = example_traj[0,:].reshape(1,2)
# end = example_traj[-1,:].reshape(1,2)

start = np.asarray([2, -2]).reshape(1,2)
end = np.asarray([2, 1]).reshape(1,2)



start = normalize_data(start, stats=stats)
end = normalize_data(end, stats=stats)


# save visualization and rewards
imgs = []
# imgs.append(map_img)

step_idx = 0
path_steps = 100
num_diffusion_iters = 1000

B = 1
# stack the last obs_horizon number of observations
images = np.stack([np.moveaxis(map_img, -1, 0) for _ in range(obs_horizon)])

# images are already normalized to [0,1]
nimages = images

# device transfer
nimages = torch.from_numpy(nimages).to(device, dtype=torch.float32)
# (1,3,100,100)
nstart = torch.from_numpy(start).to(device, dtype=torch.float32)
ngoal = torch.from_numpy(end).to(device, dtype=torch.float32)
                                                 
# infer action
with torch.no_grad():
    # get image features
    image_features = ema_nets['vision_encoder'](nimages)
    # (1,512)

    # concat with low-dim observations
    obs_features = torch.cat([image_features, nstart, ngoal], dim=-1)

    # reshape observation to (B,obs_horizon*obs_dim)
    obs_cond = obs_features.unsqueeze(0).flatten(start_dim=1)

    # initialize action from Guassian noise
    noisy_action = torch.randn(
        (B, path_steps, action_dim), device=device)

    noisy_action[0,0,:] = torch.tensor(start)
    noisy_action[0,-1,:] = torch.tensor(end)

    naction = noisy_action

    # init scheduler
    noise_scheduler.set_timesteps(num_diffusion_iters)

    for k in noise_scheduler.timesteps:

        naction[0,0,:] = torch.tensor(start)
        naction[0,-1,:] = torch.tensor(end)

        # predict noise
        noise_pred = ema_nets['noise_pred_net'](
            sample=naction,
            timestep=k,
            global_cond=obs_cond
        )

        naction[0,0,:] = torch.tensor(start)
        naction[0,-1,:] = torch.tensor(end)

        # inverse diffusion step (remove noise)
        naction = noise_scheduler.step(
            model_output=noise_pred,
            timestep=k,
            sample=naction
        ).prev_sample

        naction[0,0,:] = torch.tensor(start)
        naction[0,-1,:] = torch.tensor(end)

        # unnormalize action for visualization
        naction_np = naction.detach().to('cpu').numpy()
        naction_np = naction_np[0]
        action_pred = unnormalize_data(naction_np, stats=stats)
        
        traj_img = gen_traj_img(map_img, action_pred)
        imgs.append(traj_img)


In [ ]:
import matplotlib.animation as animation
fig = plt.figure()
import matplotlib.animation as animation

frames = [] # for storing the generated images
fig = plt.figure()
for img in imgs:
    frames.append([plt.imshow(img, animated=True)])
ani = animation.ArtistAnimation(fig, frames, interval=50, blit=True,
                                repeat=False)

ani.save('movie.mp4')

In [ ]:
# visualize
from IPython.display import Video
Video('movie.mp4', embed=True, width=760, height=760)